# Paso 2.5: Cargue de datos geoestadísticos

## Librerias

In [1]:
# Generales
import pandas as pd
import os 
import geopandas as gpd
from shapely import wkt

# Ocultar warnings
import warnings
warnings.filterwarnings('ignore')

# Aumentar número de columnas que se pueden ver
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.10f}'.format
# Cada columna será tan grande como sea necesario para mostrar todo su contenido
pd.set_option('display.max_colwidth', 0)

In [2]:
# Cambiar directorio para importar modulos y datos fácilmente
os.chdir('../')
os.getcwd()

'c:\\Users\\nrivera\\OneDrive - PROCOLOMBIA\\Documentos\\029-App-Segmentacion-Analitica\\app-segmentacion-exportaciones'

In [3]:
# Modulo de Snowflake Analítica
import src.snowflake_analitica as snowflake_analitica

2026-04-07 13:33:58.337 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 13:33:58.338 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-04-07 13:33:58.338 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 13:33:58.341 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 13:33:58.341 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 13:33:58.341 WARNING streamlit.runtime.scriptrunner_utils.script_run_c

### Snowflake

In [4]:
# Creación de sesión
json_path = './.streamlit/snowflake_credentials.json'
sesion_activa, conexion_activa = snowflake_analitica.create_session_from_json(json_file_path = json_path)
sesion_activa

## 1. DEPARTAMENTOS MGN

In [5]:
# Leer el shapefile
shapefile_path = 'data/Mapas/MGN2018_DPTO_POLITICO/MGN_DPTO_POLITICO.shp'
gdf = gpd.read_file(shapefile_path)

In [6]:
# Seleccionar columnas de interés
gdf = gdf[['DPTO_CCDGO', 'DPTO_CNMBR', 'geometry']]

In [7]:
# Convertir geometrías a WKT (Well-Known Text) para Snowflake
# Snowflake acepta GEOGRAPHY en formato WKT
gdf['GEOMETRIA_WKT'] = gdf['geometry'].apply(lambda x: x.wkt)

In [8]:
# Crear DataFrame para Snowflake
df_snowflake_departamentos = pd.DataFrame({
    'CODIGO_DEPARTAMENTO': gdf['DPTO_CCDGO'].astype(str),
    'NOMBRE_DEPARTAMENTO': gdf['DPTO_CNMBR'].astype(str),
    'GEOMETRIA': gdf['GEOMETRIA_WKT']
})

In [9]:
# Leer excel con nombres de departamentos
excel_path = 'data/Mapas/NOMBRES_DEPTO_LIMPIO.xlsx'
df_nombres_departamentos = pd.read_excel(io = excel_path, sheet_name='Hoja1', dtype={'CODIGO_DEPARTAMENTO': str, 'NOMBRE_DEPARTAMENTO_LIMPIO': str})

In [10]:
# Llenar la columna de código de departamento como string con 2 a la izquierda
df_nombres_departamentos['CODIGO_DEPARTAMENTO'] = df_nombres_departamentos['CODIGO_DEPARTAMENTO'].str.zfill(2)

In [11]:
# Unir el nombre limpio al DataFrame de departamentos
df_snowflake_departamentos = pd.merge(
    df_nombres_departamentos,
    df_snowflake_departamentos,
    left_on='CODIGO_DEPARTAMENTO',
    right_on='CODIGO_DEPARTAMENTO',
    how='left'
)

## 2. MUNICIPIOS MGN

In [12]:
# Leer el shapefile
shapefile_path = 'data/Mapas/MGN2018_MPIO_POLITICO/MGN_MPIO_POLITICO.shp'
gdf = gpd.read_file(shapefile_path)

In [13]:
gdf.columns

Index(['DPTO_CCDGO', 'MPIO_CCDGO', 'MPIO_CNMBR', 'MPIO_CRSLC', 'MPIO_NAREA',
       'MPIO_CCNCT', 'MPIO_NANO', 'DPTO_CNMBR', 'Shape_Leng', 'Shape_Area',
       'ORIG_FID', 'geometry'],
      dtype='object')

In [14]:
# Seleccionar columnas de interés
gdf = gdf[['DPTO_CCDGO', 'DPTO_CNMBR', 'MPIO_CCNCT', 'MPIO_CNMBR', 'geometry']]

In [15]:
# Convertir geometrías a WKT (Well-Known Text) para Snowflake
# Snowflake acepta GEOGRAPHY en formato WKT
gdf['GEOMETRIA_WKT'] = gdf['geometry'].apply(lambda x: x.wkt)

In [16]:
# Crear DataFrame para Snowflake
df_snowflake_municipios = pd.DataFrame({
    'CODIGO_DEPARTAMENTO': gdf['DPTO_CCDGO'].astype(str),
    'NOMBRE_DEPARTAMENTO': gdf['DPTO_CNMBR'].astype(str),
    'CODIGO_MUNICIPIO': gdf['MPIO_CCNCT'].astype(str),
    'NOMBRE_MUNICIPIO': gdf['MPIO_CNMBR'].astype(str),
    'GEOMETRIA': gdf['GEOMETRIA_WKT']
})

## 4. Geodivipola

In [56]:
# Departamentos
df_departamentos_geo_divipola = pd.read_excel('data/Mapas/GEO_DIVIPOLA.xlsx', sheet_name='Departamentos', 
                                              dtype={
                                                  'Codigo Departamento': str,
                                                  'Nombre Departamento': str,
                                                  'LATITUD': float,
                                                  'LONGITUD': float
                                              }
)

# Cambiar nombres a mayuscula y quitar espacios
df_departamentos_geo_divipola.columns = df_departamentos_geo_divipola.columns.str.upper().str.replace(' ', '_').str.replace('-', '_')

In [57]:
# Municipios
df_municipios_geo_divipola = pd.read_excel('data/Mapas/GEO_DIVIPOLA.xlsx', sheet_name='Municipios', 
                                              dtype={
                                                  'Codigo Departamento': str,
                                                  'Nombre Departamento': str,
                                                  'Codigo Municipio': str,
                                                  'Nombre Municipio': str,
                                                  'Tipo': str,
                                                  'Longitud': float,
                                                  'Latitud': float
                                              }
)

# Cambiar nombres a mayuscula y quitar espacios y caracteres especiales en los nombres de las columnas
df_municipios_geo_divipola.columns = df_municipios_geo_divipola.columns.str.upper().str.replace(' ', '_').str.replace('-', '_')

In [58]:
# Municipios
df_cabeceras_geo_divipola = pd.read_excel('data/Mapas/GEO_DIVIPOLA.xlsx', sheet_name='Cabeceras - Centros Poblados', 
                                              dtype={
                                                  'Codigo Departamento': str,
                                                  'Nombre Departamento': str,
                                                  'Codigo Municipio': str,
                                                  'Nombre Municipio': str,
                                                  'Codigo Cabecera': str,
                                                  'Nombre Cabecera': str,
                                                  'Tipo': str,
                                                  'Longitud': float,
                                                  'Latitud': float
                                              }
)

# Cambiar nombres a mayuscula y quitar espacios y caracteres especiales en los nombres de las columnas
df_cabeceras_geo_divipola.columns = df_cabeceras_geo_divipola.columns.str.upper().str.replace(' ', '_').str.replace('-', '_')

## 5. Subir a Snowflake

In [59]:
# Usar base de datos y esquema:

# Ejecutar
snowflake_analitica.update_session_params(sesion_activa, database="APP_SEGMENTACION_EXPORTACIONES", schema="PUBLIC")

Base de datos cambiada a: APP_SEGMENTACION_EXPORTACIONES
Esquema cambiado a: PUBLIC


In [60]:
# Identificar ubicación actual
snowflake_analitica.get_session_info(sesion_activa)

{'account': '"my17686"',
 'role': '"ANALYTICS"',
 'user': '"nrivera"',
 'warehouse': '"WH_PROCOLOMBIA_ANALITICA"',
 'database': '"APP_SEGMENTACION_EXPORTACIONES"',
 'schema': '"PUBLIC"'}

In [61]:
# Lista de pd a subir
bases_de_datos = [
    df_snowflake_departamentos,
    df_snowflake_municipios,
    df_departamentos_geo_divipola,
    df_municipios_geo_divipola,
    df_cabeceras_geo_divipola
]

nombres_tablas = [
    'GEOGRAFIA_DEPARTAMENTOS',
    'GEOGRAFIA_MUNICIPIOS',
    'GEOGRAFIA_DEPARTAMENTOS_GEO_DIVIPOLA',
    'GEOGRAFIA_MUNICIPIOS_GEO_DIVIPOLA',
    'GEOGRAFIA_CABECERAS_GEO_DIVIPOLA'
]

In [62]:
# Mensaje de inicio de proceso de cargue
print('Iniciando proceso de cargue...')

# pd de verificación
df_resultados_verificacion = pd.DataFrame()
# Subir y verificar bases a Snowflake
for base, tabla in zip(bases_de_datos, nombres_tablas):

    # Cambiar ubicación de la sesión para carga de datos
    snowflake_analitica.update_session_params(sesion_activa, database='APP_SEGMENTACION_EXPORTACIONES', schema='PUBLIC')

    # Obtener números de registros
    obs = len(base)
    
    # Cargar el DataFrame en Snowflake y capturar el mensaje de carga
    mensaje_carga = snowflake_analitica.upload_dataframe_to_snowflake(
            sesion_activa=sesion_activa, 
            df=base, 
            nombre_tabla=tabla, 
            create_table=True, 
            overwrite=True, 
            ram_gb=32
        )
    
    # Verificar y almacenar el resultado en el DataFrame
    resultado = sesion_activa.sql(f"SELECT COUNT(*) FROM {tabla};").collect()
    total_registros = resultado  # Extraer el total de registros

    # Crear un DataFrame temporal para la nueva fila
    nueva_fila = pd.DataFrame({
        'Tabla': [tabla],
        'Total_Registros': [total_registros],
        'Mensaje_Carga': [mensaje_carga]
    })

    # Concatenar la nueva fila al DataFrame de resultados
    df_resultados_verificacion = pd.concat([df_resultados_verificacion, nueva_fila], ignore_index=True)

    # Cambiar ubicación de la sesión para carga de datos a la tabla de auditoria
    snowflake_analitica.update_session_params(sesion_activa,  database='APP_SEGMENTACION_EXPORTACIONES', schema='SEGUIMIENTO')

     # Registrar evento de cargue
    resultado_str = '\n'.join(mensaje_carga)

    snowflake_analitica.registrar_evento_auditoria(sesion_activa=sesion_activa, 
                                                   nombre_esquema_destino='PUBLIC', 
                                                   nombre_tabla=tabla, 
                                                   numero_registros=obs, 
                                                   mensaje=resultado_str)



Iniciando proceso de cargue...
Base de datos cambiada a: APP_SEGMENTACION_EXPORTACIONES
Esquema cambiado a: PUBLIC
Base de datos cambiada a: APP_SEGMENTACION_EXPORTACIONES
Esquema cambiado a: SEGUIMIENTO
Evento de auditoría registrado con éxito.
Base de datos cambiada a: APP_SEGMENTACION_EXPORTACIONES
Esquema cambiado a: PUBLIC
Base de datos cambiada a: APP_SEGMENTACION_EXPORTACIONES
Esquema cambiado a: SEGUIMIENTO
Evento de auditoría registrado con éxito.
Base de datos cambiada a: APP_SEGMENTACION_EXPORTACIONES
Esquema cambiado a: PUBLIC
Base de datos cambiada a: APP_SEGMENTACION_EXPORTACIONES
Esquema cambiado a: SEGUIMIENTO
Evento de auditoría registrado con éxito.
Base de datos cambiada a: APP_SEGMENTACION_EXPORTACIONES
Esquema cambiado a: PUBLIC
Base de datos cambiada a: APP_SEGMENTACION_EXPORTACIONES
Esquema cambiado a: SEGUIMIENTO
Evento de auditoría registrado con éxito.
Base de datos cambiada a: APP_SEGMENTACION_EXPORTACIONES
Esquema cambiado a: PUBLIC
Base de datos cambiada a:

In [63]:
# Ver resultados
df_resultados_verificacion

,Tabla,Total_Registros,Mensaje_Carga
0,GEOGRAFIA_DEPARTAMENTOS,"[[33,]]","[Tabla 'GEOGRAFIA_DEPARTAMENTOS' creada exitosamente en Snowflake., DataFrame cargado exitosamente en la tabla 'GEOGRAFIA_DEPARTAMENTOS'., Tiempo de carga: 16.36 segundos., Proceso terminado.]"
1,GEOGRAFIA_MUNICIPIOS,"[[1182,]]","[Tabla 'GEOGRAFIA_MUNICIPIOS' creada exitosamente en Snowflake., DataFrame cargado exitosamente en la tabla 'GEOGRAFIA_MUNICIPIOS'., Tiempo de carga: 63.09 segundos., Proceso terminado.]"
2,GEOGRAFIA_DEPARTAMENTOS_GEO_DIVIPOLA,"[[33,]]","[Tabla 'GEOGRAFIA_DEPARTAMENTOS_GEO_DIVIPOLA' creada exitosamente en Snowflake., DataFrame cargado exitosamente en la tabla 'GEOGRAFIA_DEPARTAMENTOS_GEO_DIVIPOLA'., Tiempo de carga: 3.26 segundos., Proceso terminado.]"
3,GEOGRAFIA_MUNICIPIOS_GEO_DIVIPOLA,"[[1122,]]","[Tabla 'GEOGRAFIA_MUNICIPIOS_GEO_DIVIPOLA' creada exitosamente en Snowflake., DataFrame cargado exitosamente en la tabla 'GEOGRAFIA_MUNICIPIOS_GEO_DIVIPOLA'., Tiempo de carga: 3.82 segundos., Proceso terminado.]"
4,GEOGRAFIA_CABECERAS_GEO_DIVIPOLA,"[[8420,]]","[Tabla 'GEOGRAFIA_CABECERAS_GEO_DIVIPOLA' creada exitosamente en Snowflake., DataFrame cargado exitosamente en la tabla 'GEOGRAFIA_CABECERAS_GEO_DIVIPOLA'., Tiempo de carga: 4.25 segundos., Proceso terminado.]"


### Cerrar sesión, conexión y cursor

In [64]:
sesion_activa.close()